# Collocation Networks

**Computational Text Analysis Pack · Module 19 — Networks from Text**

*Technique:* Building and reading a GraphColl-style collocation network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/19a_collocation_networks.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Module 5 measured collocation pair by pair. A **collocation network** shows the
whole structure at once: words are nodes, strong associations are edges, and the
shape of the graph shows clusters of vocabulary that a ranked list flattens.

This is what **#LancsBox's GraphColl** does, and it is the most common way
corpus linguistics puts collocation on a page.

## The tooling

**networkx** builds and analyses graphs in Python; **pyvis** makes them
interactive; **igraph** and **ggraph** are the R equivalents; **Gephi** is the
standard desktop tool for laying out and styling a network by hand. In corpus
linguistics specifically, **#LancsBox** (Brezina, Lancaster) ships GraphColl,
which is a collocation network builder with the association measures already
wired in.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import networkx as nx
import math

sub = df[df["language"] == "English"]
docs = [tokenise(t) for t in sub["text"]]
unigram = Counter()
for d in docs:
    unigram.update(d)
N = sum(unigram.values())
print(f"{len(docs)} documents, {N:,} tokens")

WINDOW = 4
MIN_COUNT = 25
pairs = Counter()
for d in docs:
    for i, w in enumerate(d):
        for j in range(i + 1, min(i + WINDOW + 1, len(d))):
            if w != d[j]:
                pairs[tuple(sorted((w, d[j])))] += 1
print(f"{len(pairs):,} co-occurring pairs before filtering")

## From counts to edges

An edge should mean "these two words associate more than chance", not "these two
words are common". That is the same log-Dice decision as module 5 — the network
inherits whatever measure you choose, and choosing frequency makes a hairball of
function words.

In [ ]:
def log_dice(a, b, joint):
    return 14 + math.log2(2 * joint / (unigram[a] + unigram[b]))

STOP = {"the", "a", "an", "of", "to", "in", "and", "is", "was", "for", "on",
        "with", "at", "by", "from", "that", "this", "it", "as", "be", "are",
        "has", "have", "had", "will", "said", "he", "she", "they", "we"}

edges = []
for (a, b), joint in pairs.items():
    if joint < MIN_COUNT or a in STOP or b in STOP:
        continue
    if unigram[a] < 60 or unigram[b] < 60:
        continue
    edges.append((a, b, log_dice(a, b, joint)))
edges.sort(key=lambda e: -e[2])
edges = edges[:220]
print(f"{len(edges)} edges kept, log-Dice from "
      f"{edges[-1][2]:.2f} to {edges[0][2]:.2f}")

G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)
print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, k=0.5, seed=7, weight="weight")
deg = dict(G.degree())
nx.draw_networkx_edges(G, pos, alpha=.25, width=.8)
nx.draw_networkx_nodes(G, pos, node_size=[40 + 55 * deg[n] for n in G],
                       node_color=[deg[n] for n in G], cmap="viridis", alpha=.85)
nx.draw_networkx_labels(G, pos, font_size=7)
plt.title(f"Collocation network, log-Dice, window +/-{WINDOW}")
plt.axis("off")
plt.tight_layout()

## Reading the graph responsibly

Three measures, and what each is actually claiming.

- **Degree** — how many partners a word has. Broad, unspecific vocabulary.
- **Betweenness** — how often a word sits on the path between other words. High
  betweenness marks vocabulary that *bridges* topics, and it is the measure most
  often over-interpreted.
- **Communities** — clusters found by modularity. These are the network's version
  of topics, and like topics they need reading, not just naming.

In [ ]:
deg_c = nx.degree_centrality(G)
btw_c = nx.betweenness_centrality(G, weight=None)
print(f"{'word':<16}{'degree':>10}{'betweenness':>14}")
print("-" * 40)
for w, _ in sorted(btw_c.items(), key=lambda kv: -kv[1])[:12]:
    print(f"{w:<16}{deg_c[w]:>10.3f}{btw_c[w]:>14.4f}")
print()
communities = nx.community.greedy_modularity_communities(G)
print(f"{len(communities)} communities found\n")
for i, c in enumerate(sorted(communities, key=len, reverse=True)[:5]):
    members = sorted(c, key=lambda w: -deg[w])[:10]
    print(f"  community {i} ({len(c)} words): {', '.join(members)}")

In [ ]:
# The caution that belongs with every network figure.
print("Sensitivity check — how much does the picture depend on the cut-offs?\n")
print(f"{'min joint count':>17}{'nodes':>8}{'edges':>8}{'components':>13}")
print("-" * 46)
for mc in [10, 25, 50, 100]:
    e = [(a, b) for (a, b), j in pairs.items()
         if j >= mc and a not in STOP and b not in STOP
         and unigram[a] >= 60 and unigram[b] >= 60]
    g = nx.Graph(e[:220])
    print(f"{mc:>17}{g.number_of_nodes():>8}{g.number_of_edges():>8}"
          f"{nx.number_connected_components(g):>13}")
print()
print("A collocation network is a picture of your thresholds as much as of your")
print("corpus. Publish the window, the measure, and the cut-offs, or the figure")
print("cannot be reproduced.")

### 🔧 Your turn 1

Remove the STOP filter and redraw. What does the network look like, and what does that tell you about the filter's status?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Without the stop list the graph collapses into a dense hub of function words
with everything else hanging off it, because *the* and *of* co-occur with
everything. The filter is therefore not preprocessing hygiene — it is the
decision that produces the figure's shape, and it deserves a sentence in the
methods section. Note the tension with module 9: stylometry treats those same
function words as the entire signal. Which words are noise depends on the
question, never on the corpus.

</details>